# Chapter 18 — Can One Embedding Space Be Translated Into Another?

**Book alignment:** Embeddings From First Principles, Chapter 18

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** Fit a linear map `T` on anchor pairs so `T(E_A(x)) ≈
E_B(x)`, and test it on entities the map never saw. Does the split the chapter predicts
hold — *coarse* structure (neighbourhoods, relation ordering) survives, *fine* distinctions
(the hard-negative margin) do not? On RELATE (Wave 3), a ridge bridge from MiniLM-L6 to
mpnet-base keeps ~71% 10-NN overlap and 0.87 relation-profile correlation, but only ~23% of
the hard-negative margin.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. Fit a ridge bridge on held-out anchors — evaluate on entities it never saw

In [ ]:
n, dA, dB = 400, 48, 64
Z = rng.standard_normal((n, 12))                       # a shared latent structure
A = Z @ rng.standard_normal((12, dA)) + rng.standard_normal((n, dA)) * 0.1
B = Z @ rng.standard_normal((12, dB)) + rng.standard_normal((n, dB)) * 0.1

tr, te = slice(0, 300), slice(300, None)
lam = 1.0
W = np.linalg.solve(A[tr].T @ A[tr] + lam * np.eye(dA), A[tr].T @ B[tr])   # ridge least squares

def cos_rows(X, Y):
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    Y = Y / np.linalg.norm(Y, axis=1, keepdims=True)
    return float(np.mean(np.sum(X * Y, axis=1)))

recon_train = cos_rows(A[tr] @ W, B[tr])
recon_test  = cos_rows(A[te] @ W, B[te])
print(f"coordinate reconstruction   train {recon_train:.3f}   held-out {recon_test:.3f}")
assert recon_test < recon_train                       # scoring on train measures memorisation
print("the real question is generalisation - always report the held-out number")

## 2. The preservation matrix on RELATE (Wave 3): coarse survives, fine does not

In [ ]:
ridge = art("wave3", "ladder-8property-matrix")["pairs"]["minilm-l6 vs mpnet-base"]["rungs"]["ridge"]
print("ridge bridge, MiniLM-L6 -> mpnet-base, evaluated on unseen test entities:")
for k in ("coordinate_reconstruction", "neighborhood_at10", "relation_profile_corr",
          "rank_triplet_agreement", "calibration_transfer", "hard_negative_ratio"):
    print(f"  {k:26} {ridge[k]:.3f}")

assert ridge["neighborhood_at10"] > 0.6               # coarse: most neighbourhood structure survives
assert ridge["relation_profile_corr"] > 0.8           # coarse: the relation ORDERING stays correlated
assert ridge["hard_negative_ratio"] < 0.35            # fine: the hard-negative margin does not survive
print("\n'roughly where things go' survives a linear map; the fine distinctions do not")

## What we earned

A plain linear map recovers most *coarse* structure on unseen entities — ~71% neighbourhood
overlap, relation ordering 0.87 correlated — and keeps only ~23% of the hard-negative
margin. "≈" should be defined by downstream need (neighbours, rankings, clusters,
thresholds), not coordinate identity, and defined *before* fitting the map.

**Notebook 19 / Chapter 19** widens the map family and sharpens the reconstruction-vs-
preservation distinction.